# Partial Moments: Risk Workflow


In [1]:
import numpy as np

from pynns import (
    lpm,
    lpm_ratio,
    mean_pm,
    nns_anova,
    nns_cdf,
    nns_dep,
    nns_gravity,
    nns_mode,
    nns_rescale,
    pm_matrix,
    skew_pm,
    upm,
    upm_ratio,
    var_pm,
)

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)


## Strategy returns


In [2]:
n = 260
market = rng.normal(0.0004, 0.0090, n)
quality = 0.0007 + 0.55 * market + rng.normal(0.0, 0.0060, n)
quality[::41] -= 0.025
barbell = 0.0007 + 0.35 * market + rng.normal(0.0, 0.0110, n)
barbell[::31] -= 0.045
barbell[17::53] += 0.035
defensive = 0.00045 + 0.25 * market + rng.normal(0.0, 0.0045, n)

returns = np.column_stack((quality, barbell, defensive, market))
names = ("quality", "barbell", "defensive", "market")

def row(name: str, values: np.ndarray) -> tuple[object, ...]:
    target = 0.0
    lower2 = float(lpm(2, target, values))
    upper2 = float(upm(2, target, values))
    sortino_like = float(mean_pm(values) / np.sqrt(lower2)) if lower2 > 0 else np.nan
    return (
        name,
        mean_pm(values) * 252.0,
        np.sqrt(var_pm(values)) * np.sqrt(252.0),
        float(lpm(0, target, values)),
        lower2,
        upper2,
        sortino_like,
        skew_pm(values),
    )

print("strategy      ann_mean  ann_vol  p(loss)   LPM2@0    UPM2@0  mean/sqrt(LPM2)  skew")
for item in [row(name, returns[:, i]) for i, name in enumerate(names)]:
    print(f"{item[0]:<11} {item[1]:>8.3f} {item[2]:>8.3f} {item[3]:>8.3f} {item[4]:>9.6f} {item[5]:>9.6f} {item[6]:>15.3f} {item[7]:>7.3f}")


strategy      ann_mean  ann_vol  p(loss)   LPM2@0    UPM2@0  mean/sqrt(LPM2)  skew
quality        0.043    0.133    0.458  0.000040  0.000031           0.027  -0.669
barbell       -0.253    0.238    0.488  0.000139  0.000086          -0.085  -0.556
defensive      0.085    0.077    0.442  0.000011  0.000012           0.101  -0.187
market        -0.007    0.134    0.527  0.000032  0.000039          -0.005   0.385


## Variance decomposition


In [3]:
print("name        var_pm      LPM2(mean)+UPM2(mean)   LPM_ratio@0   UPM_ratio@0")
for i, name in enumerate(names):
    values = returns[:, i]
    center = float(np.mean(values))
    reconstructed = float(lpm(2, center, values) + upm(2, center, values))
    print(
        f"{name:<10} {var_pm(values):>9.7f} {reconstructed:>20.7f}"
        f" {float(lpm_ratio(2, 0.0, values)):>12.3f} {float(upm_ratio(2, 0.0, values)):>12.3f}"
    )


name        var_pm      LPM2(mean)+UPM2(mean)   LPM_ratio@0   UPM_ratio@0
quality    0.0000704            0.0000704        0.563        0.437
barbell    0.0002246            0.0002246        0.618        0.382
defensive  0.0000235            0.0000235        0.472        0.528
market     0.0000709            0.0000709        0.453        0.547


## Degree-zero probability checks


In [4]:
targets = np.array([-0.02, -0.01, 0.0, 0.01], dtype=np.float64)
print("targets:", targets)
for i, name in enumerate(names[:3]):
    print(f"{name:<10}", np.asarray(lpm(0, targets, returns[:, i])))

cdf = nns_cdf(quality, degree=1, target=0.0)
print("\nNNS.CDF degree=1 target value for quality:", cdf["target.value"])
print("first five CDF rows:")
fn = cdf["Function"]
print(np.column_stack((fn["x"][:5], fn["CDF"][:5])))


targets: [-0.02 -0.01  0.    0.01]
quality    [0.0269 0.1038 0.4577 0.8808]
barbell    [0.0769 0.2308 0.4885 0.7923]
defensive  [0.     0.0269 0.4423 0.9692]

NNS.CDF degree=1 target value for quality: [0.4867]
first five CDF rows:
[[-0.0306  0.    ]
 [-0.0271  0.0005]
 [-0.0254  0.001 ]
 [-0.0253  0.0011]
 [-0.0249  0.0014]]


## Partial-moment covariance


In [5]:
pm = pm_matrix(1, 1, 0.0, returns, pop_adj=True, norm=True)
print("normalized covariance-style matrix:")
print(pm["cov.matrix"])
print("\nco-lower share matrix (both assets below target together):")
print(pm["clpm"])


normalized covariance-style matrix:
[[1.     0.1016 0.2289 0.6782]
 [0.1016 1.     0.1491 0.3259]
 [0.2289 0.1491 1.     0.614 ]
 [0.6782 0.3259 0.614  1.    ]]

co-lower share matrix (both assets below target together):
[[0.5633 0.2736 0.2249 0.3532]
 [0.2736 0.6182 0.2535 0.3213]
 [0.2249 0.2535 0.4715 0.3483]
 [0.3532 0.3213 0.3483 0.4532]]


## Nonlinear dependence


In [6]:
drawdown_pressure = np.where(market < 0.0, (market * 100.0) ** 2, 0.15 * market) + rng.normal(0.0, 0.05, n)
dep = nns_dep(market, drawdown_pressure)
pearson = float(np.corrcoef(market, drawdown_pressure)[0, 1])
print("Pearson correlation:", round(pearson, 4))
print("NNS correlation/dependence:", {key: round(value, 4) for key, value in dep.items()})


Pearson correlation: -0.7089
NNS correlation/dependence: {'Correlation': 0.0499, 'Dependence': 0.6151}


## Distribution comparison


In [7]:
comparison = nns_anova(quality, barbell, confidence_interval=None)
lower_semis = np.array([float(lpm(2, 0.0, returns[:, i])) for i in range(3)])
risk_score = 100.0 - nns_rescale(lower_semis, 0.0, 100.0)

print("quality vs barbell certainty:", round(comparison["Certainty"], 4))
lower_summary = {name: round(float(value), 7) for name, value in zip(names[:3], lower_semis)}
score_summary = {name: round(float(value), 2) for name, value in zip(names[:3], risk_score)}
print("lower semivariance:", lower_summary)
print("higher-is-better downside score:", score_summary)
print("quality gravity:", round(nns_gravity(quality), 6))
print("rounded daily return mode:", nns_mode(np.round(quality, 3), discrete=True, multi=True))


quality vs barbell certainty: 0.6059
lower semivariance: {'quality': 3.97e-05, 'barbell': 0.0001394, 'defensive': 1.11e-05}
higher-is-better downside score: {'quality': 77.75, 'barbell': 0.0, 'defensive': 100.0}
quality gravity: 0.001097
rounded daily return mode: [-0.]
